# Recovered Results (Test5)

This notebook file is structurally damaged, but your **saved test5 results are still on disk**.

Run the cell below this note to load and display:
- `models/results/wikiart_tests_1_to_5_summary.csv`
- `models/results/wikiart_test5_history.csv`
- checkpoint existence for `models/wikiart_test5_best.pt`

In [1]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd()
summary_path = project_root / "models" / "results" / "wikiart_tests_1_to_5_summary.csv"
history_path = project_root / "models" / "results" / "wikiart_test5_history.csv"
ckpt_path = project_root / "models" / "wikiart_test5_best.pt"

if summary_path.exists():
    summary_df = pd.read_csv(summary_path)
    row = summary_df.loc[summary_df["experiment"].astype(str).str.lower() == "test5"]
    if not row.empty:
        cols = [c for c in ["val_top1", "test_top1", "best_epoch", "model_name", "saved_at_utc"] if c in row.columns]
        print("Recovered test5 summary:")
        display(row[cols])
    else:
        print("No test5 row found in summary CSV.")
else:
    print(f"Missing summary file: {summary_path}")

if history_path.exists():
    hist_df = pd.read_csv(history_path)
    print(f"Recovered history rows: {len(hist_df)}")
    if len(hist_df) > 0:
        print("Last 5 history rows:")
        display(hist_df.tail(5))
else:
    print(f"Missing history file: {history_path}")

print(f"Checkpoint exists: {ckpt_path.exists()} -> {ckpt_path}")

Recovered test5 summary:


,val_top1,test_top1,best_epoch,model_name,saved_at_utc
4,0.692986,0.681136,20.0,vit_base_patch16_384,2026-03-12T14:07:50.261156+00:00


Recovered history rows: 20
Last 5 history rows:


,epoch,stage,lr,train_loss,train_top1,train_top5,val_loss,val_top1,val_top5
15,16,fine-tune,2.196759e-06,3.375039,0.713046,0.846483,5.624681,0.686434,0.798246
16,17,fine-tune,1.611289e-06,3.370603,0.721849,0.854024,5.629701,0.689903,0.795211
17,18,fine-tune,1.136379e-06,3.362761,0.720341,0.852323,5.635659,0.689228,0.793044
18,19,fine-tune,7.864601e-07,3.356550,0.733318,0.858057,5.639787,0.690384,0.790876
19,20,fine-tune,5.721632e-07,3.356220,0.730758,0.862319,5.637950,0.692986,0.792369


Checkpoint exists: True -> c:\Users\Thijs\Desktop\Ai Art Critic\models\wikiart_test5_best.pt


# WikiArt Style Classification - Test 4 (Improved Pipeline)

This notebook is a stronger experimental version of `test3`.

Main upgrades in this version:
- two-stage training (freeze backbone -> fine-tune all layers)
- stronger backbone (`ConvNeXt-Tiny`)
- class-weighted loss for imbalance
- stronger augmentation
- learning-rate scheduler + early stopping
- Top-1 and Top-5 accuracy tracking

## 1. Project Setup and Imports

We keep the setup readable and reproducible with fixed random seeds.

## 2. Dataset Loading and Verification

This section reuses the working auto-discovery approach from the previous notebook.

In [3]:
def extract_style_name(relative_path: str) -> str:
    return Path(relative_path).parts[0]

train_df["style_name"] = train_df["relative_path"].map(extract_style_name)
val_df["style_name"] = val_df["relative_path"].map(extract_style_name)

label_to_style = (
    train_df[["label", "style_name"]]
    .drop_duplicates()
    .sort_values("label")
    .set_index("label")["style_name"]
    .to_dict()
)

num_classes = len(label_to_style)
print(f"Number of classes: {num_classes}")
print("First 10 mappings:")
for lbl, style in list(label_to_style.items())[:10]:
    print(f"{lbl:2d} -> {style}")

In [4]:
image_size = 224
batch_size = 24

# NOTE: In Windows notebooks, worker processes can add major overhead or appear to stall.
# Keep this conservative for stable throughput.
if device.type == "cuda":
    if os.name == "nt":
        num_workers = 0
    else:
        num_workers = min(8, max(2, (os.cpu_count() or 4) // 2))
else:
    num_workers = 0

pin_memory = device.type == "cuda"
persistent_workers = num_workers > 0
use_amp = device.type == "cuda"

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(image_size, scale=(0.65, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandAugment(num_ops=2, magnitude=9),
    transforms.RandomAffine(degrees=0, translate=(0.08, 0.08), scale=(0.9, 1.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
    transforms.RandomErasing(p=0.20, scale=(0.02, 0.18), ratio=(0.3, 3.3), value="random"),
])

eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(image_size),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

print(f"image_size={image_size}, batch_size={batch_size}, num_workers={num_workers}, pin_memory={pin_memory}, persistent_workers={persistent_workers}, use_amp={use_amp}")

Project root: C:\Users\Thijs\Desktop\Ai Art Critic
WikiArt dir:  C:\Users\Thijs\Desktop\Ai Art Critic\datasets\Wikiart
Train CSV:    style_train.csv
Val CSV:      style_val.csv
Train rows:   57,025
Val rows:     24,421


,relative_path,label
0,Impressionism/edgar-degas_landscape-on-the-orn...,12
1,Realism/camille-corot_mantes-cathedral.jpg,21
2,Abstract_Expressionism/gene-davis_untitled-197...,0
3,Symbolism/kuzma-petrov-vodkin_in-the-1920.jpg,24
4,Impressionism/maurice-prendergast_paris-boulev...,12


In [3]:
class WikiArtStyleDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, image_root: Path, transform=None):
        self.df = dataframe.reset_index(drop=True).copy()
        self.image_root = Path(image_root)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self.image_root / row["relative_path"]
        label = int(row["label"])

        image = Image.open(img_path).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)

        return image, label

Number of classes: 27
First 10 mappings:
 0 -> Abstract_Expressionism
 1 -> Action_painting
 2 -> Analytical_Cubism
 3 -> Art_Nouveau_Modern
 4 -> Baroque
 5 -> Color_Field_Painting
 6 -> Contemporary_Realism
 7 -> Cubism
 8 -> Early_Renaissance
 9 -> Expressionism


## 5. Model Setup (Fast + Strong)

We keep a strong pretrained backbone but tune hyperparameters for better **accuracy-per-time**.

Key differences from test4:
- fewer epochs + earlier unfreeze
- cosine LR decay instead of plateau-only reduction
- exponential moving average (EMA) weights for more stable validation performance

## 6. Training Improvements

This notebook uses:
- two-stage fine-tuning (head -> full model)
- class-weighted label-smoothed cross-entropy
- MixUp/CutMix regularization
- cosine LR schedule
- EMA model averaging
- early stopping on validation Top-1

## 7. Training Loop

We track per-epoch metrics:
- train/validation loss
- train/validation Top-1 accuracy
- train/validation Top-5 accuracy

We also save the best checkpoint and restore it at the end.

In [10]:
# Optional training curves (only if matplotlib is available)
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print("matplotlib is not installed in this environment. Skipping plots.")

if plt is not None and len(history_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(history_df["epoch"], history_df["train_loss"], label="train")
    axes[0].plot(history_df["epoch"], history_df["val_loss"], label="val")
    axes[0].set_title("Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    axes[1].plot(history_df["epoch"], history_df["train_top1"], label="train_top1")
    axes[1].plot(history_df["epoch"], history_df["val_top1"], label="val_top1")
    axes[1].plot(history_df["epoch"], history_df["val_top5"], label="val_top5")
    axes[1].set_title("Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()

    plt.tight_layout()
    plt.show()

In [11]:
def evaluate_model(model, loader, criterion):
    loss, top1, top5 = run_one_epoch(model, loader, criterion, optimizer=None)
    return {"loss": loss, "top1": top1, "top5": top5}

val_metrics = evaluate_model(model, val_loader, criterion)
print(
    f"Validation -> loss: {val_metrics['loss']:.4f}, "
    f"top1: {val_metrics['top1']:.3f}, top5: {val_metrics['top5']:.3f}"
)

if test_loader is not None:
    test_metrics = evaluate_model(model, test_loader, criterion)
    print(
        f"Test       -> loss: {test_metrics['loss']:.4f}, "
        f"top1: {test_metrics['top1']:.3f}, top5: {test_metrics['top5']:.3f}"
    )

train: kept 57,023, removed 2 bad rows (took 4.3s)
validation: kept 24,421, removed 0 bad rows (took 1.6s)
Train samples: 57,023
Val samples:   20,758
Test samples:  3,663 (split from validation)


In [12]:
def predict_image_style(model, image_path: Path, transform, label_to_style_map, topk=5):
    model.eval()
    image = Image.open(image_path).convert("RGB")
    tensor = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(tensor)
        probs = torch.softmax(logits, dim=1)
        top_probs, top_labels = probs.topk(k=min(topk, probs.shape[1]), dim=1)

    top_labels = top_labels[0].tolist()
    top_probs = top_probs[0].tolist()
    top_styles = [label_to_style_map.get(lbl, f"Unknown {lbl}") for lbl in top_labels]
    return top_labels[0], top_styles[0], list(zip(top_styles, top_probs))

source_df = test_df if test_df is not None and len(test_df) > 0 else val_eval_df
sample_df = source_df.sample(n=min(5, len(source_df)), random_state=SEED).reset_index(drop=True)

for i, row in sample_df.iterrows():
    sample_relative_path = row["relative_path"]
    true_label = int(row["label"])
    true_style = label_to_style.get(true_label, f"Unknown {true_label}")

    pred_label, pred_style, top5 = predict_image_style(
        model, wikiart_dir / sample_relative_path, eval_transform, label_to_style, topk=5
    )

    print(f"\nExample {i + 1}")
    print(f"Image path:       {sample_relative_path}")
    print(f"True style:       {true_style} ({true_label})")
    print(f"Predicted style:  {pred_style} ({pred_label})")
    print("Top-5 predictions:")
    for style_name, prob in top5:
        print(f"  - {style_name}: {prob:.3f}")

In [13]:
# Quick summary of final training results
print("History rows:", len(history_df))
if len(history_df) > 0:
    print("\nBest validation Top-1 row:")
    best_idx = history_df["val_top1"].idxmax()
    display(history_df.loc[[best_idx]])

    print("\nLast 5 epochs:")
    display(history_df.tail(5))

print("\nFinal evaluation metrics:")
print("Validation:", val_metrics)
if "test_metrics" in globals():
    print("Test:", test_metrics)

print("\nBest checkpoint path:", checkpoint_path)
print("Best epoch:", best_epoch)
print("Best val_top1:", best_val_top1)

Model: resnet50
FAST_ITERATION_MODE=True, TOTAL_EPOCHS=8, HEAD_EPOCHS=2
MAX_TRAIN_BATCHES=40, MAX_VAL_BATCHES=20
HEAD_LR=0.001, FT_LR=0.0003, WEIGHT_DECAY=0.0001, LABEL_SMOOTHING=0.05
EMA_DECAY=0.999


Model: convnext_tiny
FAST_ITERATION_MODE=False, TOTAL_EPOCHS=24, HEAD_EPOCHS=3
MAX_TRAIN_BATCHES=None, MAX_VAL_BATCHES=None
HEAD_LR=0.001, FT_LR=0.0002, WEIGHT_DECAY=8e-05, LABEL_SMOOTHING=0.08
EMA_DECAY=0.999


model.safetensors:   0%|          | 0.00/347M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/347M [00:00<?, ?B/s]

Using pretrained variant: vit_base_patch16_384.augreg_in21k_ft_in1k
Model: vit_base_patch16_384
FAST_ITERATION_MODE=False, TOTAL_EPOCHS=20, HEAD_EPOCHS=2
MAX_TRAIN_BATCHES=None, MAX_VAL_BATCHES=None
HEAD_LR=0.0006, FT_BACKBONE_LR=1e-05, FT_HEAD_LR=3e-05, WEIGHT_DECAY=8e-05, LABEL_SMOOTHING=0.15
EMA_DECAY=0.999


In [14]:
import json
import re
from datetime import datetime, timezone


def _collect_output_texts(cell):
    texts = []
    for out in cell.get("outputs", []):
        out_type = out.get("output_type")
        if out_type == "stream":
            t = out.get("text", "")
            if isinstance(t, list):
                t = "".join(t)
            texts.append(str(t))
        elif out_type in {"execute_result", "display_data"}:
            data = out.get("data", {})
            for key in ["text/plain", "text/markdown"]:
                if key in data:
                    t = data[key]
                    if isinstance(t, list):
                        t = "".join(t)
                    texts.append(str(t))
    return "\n".join(texts)


def _extract_last_float(text, pattern):
    matches = re.findall(pattern, text, flags=re.IGNORECASE)
    if not matches:
        return None
    return float(matches[-1])


def _extract_last_epoch_line_metrics(text):
    metrics = {"val_loss": None, "val_top1": None}
    epoch_matches = re.findall(
        r"Epoch\s+\d+/\d+\s*\|[^\n]*?val_loss\s*=\s*([0-9.]+)\s*,\s*val_acc\s*=\s*([0-9.]+)",
        text,
        flags=re.IGNORECASE,
    )
    if epoch_matches:
        val_loss, val_acc = epoch_matches[-1]
        metrics["val_loss"] = float(val_loss)
        metrics["val_top1"] = float(val_acc)
    return metrics


def _parse_metric_block(text):
    metrics = {
        "val_loss": None,
        "val_top1": None,
        "val_top5": None,
        "test_loss": None,
        "test_top1": None,
        "test_top5": None,
        "best_val_top1": None,
        "best_epoch": None,
    }

    val_pat = re.search(
        r"Validation\s*->\s*loss:\s*([0-9.]+)\s*,\s*top1:\s*([0-9.]+)\s*,\s*top5:\s*([0-9.]+)",
        text,
        flags=re.IGNORECASE,
    )
    if val_pat:
        metrics["val_loss"] = float(val_pat.group(1))
        metrics["val_top1"] = float(val_pat.group(2))
        metrics["val_top5"] = float(val_pat.group(3))

    test_pat = re.search(
        r"Test\s*->\s*loss:\s*([0-9.]+)\s*,\s*top1:\s*([0-9.]+)\s*,\s*top5:\s*([0-9.]+)",
        text,
        flags=re.IGNORECASE,
    )
    if test_pat:
        metrics["test_loss"] = float(test_pat.group(1))
        metrics["test_top1"] = float(test_pat.group(2))
        metrics["test_top5"] = float(test_pat.group(3))

    if metrics["val_top1"] is None:
        metrics["val_top1"] = _extract_last_float(
            text,
            r"(?:Best\s+validation\s+accuracy|Validation\s+accuracy|Final\s+validation\s+accuracy)\s*:\s*([0-9.]+)",
        )

    if metrics["test_top1"] is None:
        metrics["test_top1"] = _extract_last_float(
            text,
            r"(?:Test\s+accuracy\s*\(from\s+val\s+split\)|Simulated\s+test\s+accuracy|Final\s+test\s+accuracy)\s*:\s*([0-9.]+)",
        )

    if metrics["val_loss"] is None or metrics["val_top1"] is None:
        ep = _extract_last_epoch_line_metrics(text)
        if metrics["val_loss"] is None:
            metrics["val_loss"] = ep["val_loss"]
        if metrics["val_top1"] is None:
            metrics["val_top1"] = ep["val_top1"]

    metrics["best_val_top1"] = _extract_last_float(
        text,
        r"(?:Best\s+val_top1|Best\s+validation\s+accuracy)\s*:\s*([0-9.]+)",
    )
    metrics["best_epoch"] = _extract_last_float(text, r"Best\s+epoch\s*:\s*([0-9]+)")

    if metrics["val_top5"] is None and metrics["val_top1"] is not None:
        metrics["val_top5"] = metrics["val_top1"]
    if metrics["test_top5"] is None and metrics["test_top1"] is not None:
        metrics["test_top5"] = metrics["test_top1"]

    return metrics


def _extract_model_name(notebook_cells, text_blob):
    joined = "\n".join(
        "\n".join(c.get("source", []))
        for c in notebook_cells
        if c.get("cell_type") == "code"
    )
    m = re.search(r'MODEL_NAME\s*=\s*["\']([^"\']+)["\']', joined)
    if m:
        return m.group(1)

    for name in ["convnext_tiny", "resnet50", "resnet34", "resnet18"]:
        if re.search(name, joined, flags=re.IGNORECASE) or re.search(name, text_blob, flags=re.IGNORECASE):
            return name
    return None


def parse_notebook_metrics(nb_path: Path):
    if not nb_path.exists():
        return {
            "notebook": nb_path.name,
            "exists": False,
            "model_name": None,
            "val_loss": None,
            "val_top1": None,
            "val_top5": None,
            "test_loss": None,
            "test_top1": None,
            "test_top5": None,
            "best_val_top1": None,
            "best_epoch": None,
        }

    with nb_path.open("r", encoding="utf-8") as f:
        nb = json.load(f)

    cells = nb.get("cells", [])
    text_blob = "\n".join(_collect_output_texts(c) for c in cells if c.get("cell_type") == "code")
    parsed = _parse_metric_block(text_blob)

    return {
        "notebook": nb_path.name,
        "exists": True,
        "model_name": _extract_model_name(cells, text_blob),
        **parsed,
    }


rows = []
for i in range(1, 6):
    nb_path = project_root / f"wikiart_style_classification_test{i}.ipynb"
    row = parse_notebook_metrics(nb_path)
    row["experiment"] = f"test{i}"
    rows.append(row)

for row in rows:
    if row["experiment"] == "test5":
        row["model_name"] = MODEL_NAME if "MODEL_NAME" in globals() else row["model_name"]
        if "val_metrics" in globals():
            row["val_loss"] = float(val_metrics.get("loss", np.nan))
            row["val_top1"] = float(val_metrics.get("top1", np.nan))
            row["val_top5"] = float(val_metrics.get("top5", np.nan))
        if "test_metrics" in globals():
            row["test_loss"] = float(test_metrics.get("loss", np.nan))
            row["test_top1"] = float(test_metrics.get("top1", np.nan))
            row["test_top5"] = float(test_metrics.get("top5", np.nan))
        if "best_val_top1" in globals():
            row["best_val_top1"] = float(best_val_top1)
        if "best_epoch" in globals():
            row["best_epoch"] = int(best_epoch)

run_timestamp = datetime.now(timezone.utc).isoformat()
for row in rows:
    row["saved_at_utc"] = run_timestamp

results_df = pd.DataFrame(rows).sort_values("experiment").reset_index(drop=True)

results_dir = project_root / "models" / "results"
results_dir.mkdir(parents=True, exist_ok=True)

summary_csv = results_dir / "wikiart_tests_1_to_5_summary.csv"
summary_json = results_dir / "wikiart_tests_1_to_5_summary.json"
test5_history_csv = results_dir / "wikiart_test5_history.csv"

tmp_csv = summary_csv.with_suffix(".csv.tmp")
results_df.to_csv(tmp_csv, index=False)
os.replace(tmp_csv, summary_csv)

tmp_json = summary_json.with_suffix(".json.tmp")
with tmp_json.open("w", encoding="utf-8") as f:
    json.dump(results_df.to_dict(orient="records"), f, indent=2)
os.replace(tmp_json, summary_json)

if "history_df" in globals() and len(history_df) > 0:
    tmp_hist = test5_history_csv.with_suffix(".csv.tmp")
    history_df.to_csv(tmp_hist, index=False)
    os.replace(tmp_hist, test5_history_csv)

print("Saved consolidated results:")
print(f" - {summary_csv}")
print(f" - {summary_json}")
if "history_df" in globals() and len(history_df) > 0:
    print(f" - {test5_history_csv}")

display(results_df)